In [1]:
# reload magics
%load_ext autoreload
%autoreload 2

In [2]:
from utils.experiment_utils import get_all_experiments_info, load_best_model
import torch
import os
import hydra
from omegaconf import DictConfig, OmegaConf

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

import ot

from datasets.lineage_tracing import LTSeqDataset

from geomloss import SamplesLoss

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from utils.eval_utils import compute_mmd_distance

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [3]:
lts = LTSeqDataset(seed=42)

loading cached adata from ./data/processed/adata_pca_50.h5ad  !!
loading cached clone sets from ./data/processed!
splitting 1218 clones into 609 train and 609 test


In [4]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'lineage' in c['name'] and 'reg' not in c['name']]

fm_models = [c for c in cfgs if 'Flow' in c['generator']]
energy_models = [c for c in cfgs if 'mmd' in c['config']['generator'].values()]
sw_models = [c for c in cfgs if 'swd' in c['config']['generator'].values()]

print('fm models: ', fm_models)
print('energy models: ', energy_models)
print('sw models: ', sw_models)

fm models:  [{'name': 'lineage_supervised_8f559753c149d773fa8c49a89dcdf1a9', 'dir': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/lineage_supervised_8f559753c149d773fa8c49a89dcdf1a9', 'config': {'dataset': {'_target_': 'datasets.lineage_tracing.LTSeqDataset', 'set_size': 100, 'min_cells': 3, 'data_shape': [50], 'root': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/data', 'seed': '${seed}'}, 'encoder': {'_target_': 'encoder.encoders.DistributionEncoderGNN', 'in_dim': '${dataset.data_shape[0]}', 'latent_dim': '${experiment.latent_dim}', 'hidden_dim': '${experiment.hidden_dim}', 'set_size': '${experiment.set_size}', 'layers': 2, 'fc_layers': 2}, 'model': {'_target_': 'layers.MLP', 'in_dims': [50, 1, 128, 128], 'hidden_dim': 512, 'out_dim': 50, 'layers': 4}, 'coupling': {'_target_': 'types.NoneType'}, 'generator': {'_target_': 'generator.flow_matching.FlowMatchingGenerator', 'model': '${model}', 'sigma': 0.5}, 'optimizer': {'_target_': 'torch

In [5]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

def sliced_wasserstein_distance(x, y, num_projections=50, p=2):
    d = x.shape[1]
    
    # Generate random directions on the unit sphere
    theta = torch.randn(d, num_projections, device=x.device)
    theta = theta / torch.norm(theta, dim=0, keepdim=True)
    
    # Project samples onto each direction
    x_proj = x @ theta  # (n, num_projections)
    y_proj = y @ theta  # (m, num_projections)
    
    # Sort projections
    x_sorted = torch.sort(x_proj, dim=0)[0]
    y_sorted = torch.sort(y_proj, dim=0)[0]
    
    # Compute 1D Wasserstein distances
    if x_sorted.shape[0] != y_sorted.shape[0]:
        min_size = min(x_sorted.shape[0], y_sorted.shape[0])
        x_quantiles = x_sorted[torch.linspace(0, x_sorted.shape[0]-1, min_size, device=x.device).long()]
        y_quantiles = y_sorted[torch.linspace(0, y_sorted.shape[0]-1, min_size, device=y.device).long()]
        wasserstein_dists = torch.mean(torch.abs(x_quantiles - y_quantiles) ** p, dim=0) ** (1/p)
    else:
        wasserstein_dists = torch.mean(torch.abs(x_sorted - y_sorted) ** p, dim=0) ** (1/p)
    
    # Average over all projections
    return torch.mean(wasserstein_dists)

loss_energy = SamplesLoss("energy")
# loss_mmd = SamplesLoss("gaussian", blur=1)

def evaluate_model(y_hat, y_true, model_name):
    """Compute energy distance, MMD, and Sliced Wasserstein."""
    energy = loss_energy(y_hat, y_true)
    mmd = compute_mmd_distance(y_hat, y_true)
    
    # Compute sliced wasserstein for each sample pair
    sw_distances = []
    for i in range(y_hat.shape[0]):
        sw = sliced_wasserstein_distance(y_hat[i], y_true[i], num_projections=50)
        sw_distances.append(sw.item())
    sw_distances = np.array(sw_distances)
    
    energy_mean = energy.mean().item()
    energy_sem = energy.std().item() / np.sqrt(energy.shape[0])
    
    mmd_mean = mmd.mean().item()
    mmd_sem = mmd.std().item() / np.sqrt(mmd.shape[0])
    
    sw_mean = sw_distances.mean()
    sw_sem = sw_distances.std() / np.sqrt(len(sw_distances))
    
    print(f"{model_name}")
    print(f"Energy Distance: {energy_mean:.6f} \pm {energy_sem:.6f}")
    print(f"MMD (RBF kernel):  {mmd_mean:.6f} \pm {mmd_sem:.6f}")
    print(f"Sliced Wasserstein: {sw_mean:.6f} \pm {sw_sem:.6f}")
    print()
    
    return {
        'energy': (energy_mean, energy_sem),
        'mmd': (mmd_mean, mmd_sem),
        'sliced_wasserstein': (sw_mean, sw_sem)
    }

In [6]:
import torch
import torch.nn as nn
import numpy as np


class ConditionalSampler(nn.Module):
    def __init__(self, x_dim, z_dim, y_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim + z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, y_dim),
        )
        self.z_dim = z_dim

    def forward(self, x, z=None):
        if z is None:
            z = torch.randn(x.shape[0], self.z_dim, device=x.device)
        return self.net(torch.cat([x, z], dim=-1))

    def sample(self, x, k=1):
        # returns (k, n, y_dim)
        n = x.shape[0]
        x_rep = x.unsqueeze(0).expand(k, -1, -1).reshape(k * n, -1)
        z = torch.randn(k * n, self.z_dim, device=x.device)
        return self.net(torch.cat([x_rep, z], dim=-1)).reshape(k, n, -1)


def energy_score_loss(samples, y_true, k=8):
    # samples: (k, n, d), y_true: (n, d)
    # ES = E||Y' - y|| - 0.5 * E||Y' - Y''||
    diff_to_true = torch.cdist(samples.permute(1, 0, 2), y_true.unsqueeze(1))  # (n, k, 1)
    term1 = diff_to_true.squeeze(-1).mean(dim=1)  # (n,)

    # pairwise distances between samples
    pairwise = torch.cdist(samples.permute(1, 0, 2), samples.permute(1, 0, 2))  # (n, k, k)
    term2 = pairwise.sum(dim=(1, 2)) / (k * (k - 1) + 1e-8)  # (n,)

    return (term1 - 0.5 * term2).mean()


def fit_conditional_sampler(X_train, Y_train, z_dim=16, hidden=256,
                            lr=1e-3, epochs=500, batch_size=256, k=8, device='cuda'):
    x_dim = X_train.shape[1]
    y_dim = Y_train.shape[1]
    model = ConditionalSampler(x_dim, z_dim, y_dim, hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    X = torch.tensor(X_train, dtype=torch.float32, device=device)
    Y = torch.tensor(Y_train, dtype=torch.float32, device=device)
    n = X.shape[0]

    for epoch in range(epochs):
        idx = torch.randint(0, n, (batch_size,), device=device)
        xb, yb = X[idx], Y[idx]
        samples = model.sample(xb, k=k)  # (k, batch, y_dim)
        loss = energy_score_loss(samples, yb, k=k)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1}/{epochs}  loss={loss.item():.4f}")

    return model

In [7]:
def evaluate_model_distributional(model_configs, lts, device='cuda', k_max=64, k_values=(1, 4, 16, 64), scale=1.0):
    """
    Evaluate a triplet of models (semisupervised, supervised, oracle),
    using distributional regression in the latent space.
    
    Samples k_max latent targets per test point once, then evaluates
    at each k in k_values by taking the per-example min over the first k samples.
    """

    enc_semi, gen_semi = load_model(
        model_configs['semisupervised']['config'], 
        model_configs['semisupervised']['dir'], 
        device
    )
    enc_sup, gen_sup = load_model(
        model_configs['supervised']['config'], 
        model_configs['supervised']['dir'], 
        device
    )
    
    # encode with semisupervised model
    with torch.no_grad():
        z_x_train = enc_semi(lts.train_srcs.to(device))
        z_y_train = enc_semi(lts.train_tgts.to(device))
        z_x_test = enc_semi(lts.test_srcs.to(device))
        z_y_test = enc_semi(lts.test_tgts.to(device))
    
    # encode with supervised model
    with torch.no_grad():
        z_x_sup_test = enc_sup(lts.test_srcs.to(device))

    X_train = z_x_train.cpu().numpy()
    Y_train = z_y_train.cpu().numpy()
    X_test = z_x_test.cpu().numpy()
    Y_test = z_y_test.cpu().numpy()
    
    src_test = lts.test_srcs
    tgt_test = lts.test_tgts
    n_test = X_test.shape[0]

    sampler = fit_conditional_sampler(X_train, Y_train, z_dim=X_train.shape[1], hidden=128,
                                      epochs=500, k=k_max, device=device)
    with torch.no_grad():
        Y_pred_samples = sampler.sample(
            torch.tensor(X_test, dtype=torch.float32, device=device), k=k_max
        ).cpu().numpy()  # (k_max, n_test, d)
    
    # --- generate predictions for all k_max samples at once ---
    semi_preds = []
    for i in range(k_max):
        z_y_pred_i = torch.tensor(Y_pred_samples[i], dtype=torch.float32).to(device)
        with torch.no_grad():
            y_hat_i = gen_semi.sample(
                src_test.to(device).reshape(-1, 50),
                z_x_test.to(device),
                z_y_pred_i
            ).reshape(tgt_test.shape)
        semi_preds.append(y_hat_i)
    
    # stack: (k_max, n_test, ...)
    semi_preds = torch.stack(semi_preds, dim=0)

    # --- compute per-example metrics for all k_max samples once ---
    n = tgt_test.shape[0]
    sw_all = np.zeros((k_max, n))
    energy_all = np.zeros((k_max, n))
    mmd_all = np.zeros((k_max, n))

    for i in range(k_max):
        y_hat_i = semi_preds[i]
        y_true = tgt_test.to(device)

        energy_all[i] = loss_energy(y_hat_i, y_true).cpu().numpy()
        mmd_all[i] = compute_mmd_distance(y_hat_i, y_true).cpu().numpy()

        for j in range(n):
            sw_all[i, j] = sliced_wasserstein_distance(
                y_hat_i[j], y_true[j], num_projections=50
            ).item()

    # --- evaluate at each k by taking min over first k samples ---
    semi_results = {}
    for k in k_values:
        assert k <= k_max, f"k={k} exceeds k_max={k_max}"
        energy_best = energy_all[:k].min(axis=0)
        mmd_best = mmd_all[:k].min(axis=0)
        sw_best = sw_all[:k].min(axis=0)

        energy_mean, energy_sem = energy_best.mean(), energy_best.std() / np.sqrt(n)
        mmd_mean, mmd_sem = mmd_best.mean(), mmd_best.std() / np.sqrt(n)
        sw_mean, sw_sem = sw_best.mean(), sw_best.std() / np.sqrt(n)

        print(f"Semi-supervised (best of k={k})")
        print(f"  Energy Distance:    {energy_mean:.6f} ± {energy_sem:.6f}")
        print(f"  MMD (RBF kernel):   {mmd_mean:.6f} ± {mmd_sem:.6f}")
        print(f"  Sliced Wasserstein: {sw_mean:.6f} ± {sw_sem:.6f}")
        print()

        semi_results[k] = {
            'energy': (energy_mean, energy_sem),
            'mmd': (mmd_mean, mmd_sem),
            'sliced_wasserstein': (sw_mean, sw_sem),
        }
    
    # oracle with true embeddings (unchanged)
    y_hat_oracle = gen_semi.sample(
        src_test.to(device).reshape(-1, 50),
        z_x_test.to(device),
        z_y_test.to(device)
    ).reshape(tgt_test.shape)
    
    # supervised model (unchanged)
    y_hat_sup = gen_sup.sample(
        src_test.to(device).reshape(-1, 50),
        z_x_sup_test.to(device),
        torch.zeros(z_x_test.shape).to(device)
    ).reshape(tgt_test.shape)

    model_name = model_configs['name']
    if model_name == 'DirectGenerator':
        model_name = model_configs['supervised']['config']['generator']['loss_type']
    
    results = {
        'model_name': model_name,
        'k_values': list(k_values),
        'scale': scale,
        'semisupervised': semi_results,
        'oracle': evaluate_model(y_hat_oracle.to(device), tgt_test.to(device), 
                                 f"{model_name} - Oracle"),
        'supervised': evaluate_model(y_hat_sup.to(device), tgt_test.to(device), 
                                    f"{model_name} - Supervised")
    }
    
    return results


# organize configs into pairs
def pair_models(model_list):
    
    semi_models = [m for m in model_list if 'semisupervised' in m['name']]
    sup_models = [m for m in model_list if '_supervised' in m['name']]

    print(semi_models)
    
    if len(semi_models) > 1 or len(sup_models) > 1:
        print('warning multiple configs found, taking first only')
    pair = {'name': semi_models[0]['generator'].split('.')[-1],
            'semisupervised': semi_models[0],
            'supervised': sup_models[0]} 
    
    return [pair]


fm_pairs = pair_models(fm_models)
energy_pairs = pair_models(energy_models)
sw_pairs = pair_models(sw_models)

all_pairs = fm_pairs + energy_pairs + sw_pairs

print(f"found {len(all_pairs)} model pairs to evaluate:")
for pair in all_pairs:
    print(f"  - {pair['name']}: {pair['semisupervised']['name']} and {pair['supervised']['name']}")
print()

# Run evaluation on all pairs
all_results = []
for pair in all_pairs:
    print(f"evaluating {pair['name']}")
    results = evaluate_model_distributional(pair, lts, k_max=32, k_values=(1, 2, 4, 8, 16, 32), device='cuda')
    all_results.append(results)
    print()

[{'name': 'lineage_semisupervised_43320c8fa9a07dd024cf61c9851479e7', 'dir': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/lineage_semisupervised_43320c8fa9a07dd024cf61c9851479e7', 'config': {'dataset': {'_target_': 'datasets.lineage_tracing.LTSeqDatasetUnstructured', 'set_size': 100, 'min_cells': 3, 'data_shape': [50], 'root': '/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/data', 'seed': '${seed}'}, 'encoder': {'_target_': 'encoder.encoders.DistributionEncoderGNN', 'in_dim': '${dataset.data_shape[0]}', 'latent_dim': '${experiment.latent_dim}', 'hidden_dim': '${experiment.hidden_dim}', 'set_size': '${experiment.set_size}', 'layers': 2, 'fc_layers': 2}, 'model': {'_target_': 'layers.MLP', 'in_dims': [50, 1, 128, 128], 'hidden_dim': 512, 'out_dim': 50, 'layers': 4}, 'coupling': {'_target_': 'types.NoneType'}, 'generator': {'_target_': 'generator.flow_matching.FlowMatchingGenerator', 'model': '${model}', 'sigma': 0.5}, 'optimizer': {'_target_'

In [10]:
def gen_label(name):
    if 'Flow' in name:
        return 'Flow matching'
    if 'mmd' in name:
        return 'Energy'
    return 'Sliced Wasserstein'

def fmt_mean_sem(t):
    return f"${t[0]:.3f} \\pm {t[1]:.3f}$"

k_values = [1, 2, 4, 8, 16, 32]

rows = []
for r in all_results:
    g = gen_label(r['model_name'])
    row = {'Generator': g}
    for k in k_values:
        row[f'$k={k}$'] = fmt_mean_sem(r['semisupervised'][k]['energy'])
    row['Oracle'] = fmt_mean_sem(r['oracle']['energy'])
    row['Ridge'] = ''
    rows.append(row)

df = pd.DataFrame(rows)
gen_order = ['Energy', 'Sliced Wasserstein', 'Flow matching']
df['Generator'] = pd.Categorical(df['Generator'], categories=gen_order, ordered=True)
df = df.sort_values('Generator').reset_index(drop=True)
df_indexed = df.set_index('Generator')

display(df_indexed.style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]},
]))

# latex
latex = df_indexed.to_markdown()
print(latex)

,$k=1$,$k=2$,$k=4$,$k=8$,$k=16$,$k=32$,Oracle,Ridge
Generator,,,,,,,,
Energy,$6.423 \pm 0.150$,$5.100 \pm 0.115$,$4.306 \pm 0.098$,$3.782 \pm 0.081$,$3.440 \pm 0.072$,$3.177 \pm 0.067$,$1.440 \pm 0.022$,
Sliced Wasserstein,$6.556 \pm 0.148$,$5.279 \pm 0.124$,$4.529 \pm 0.105$,$3.958 \pm 0.093$,$3.578 \pm 0.083$,$3.271 \pm 0.077$,$1.494 \pm 0.025$,
Flow matching,$6.311 \pm 0.141$,$5.172 \pm 0.113$,$4.462 \pm 0.100$,$3.944 \pm 0.088$,$3.606 \pm 0.076$,$3.393 \pm 0.074$,$2.539 \pm 0.052$,


| Generator          | $k=1$             | $k=2$             | $k=4$             | $k=8$             | $k=16$            | $k=32$            | Oracle            | Ridge   |
|:-------------------|:------------------|:------------------|:------------------|:------------------|:------------------|:------------------|:------------------|:--------|
| Energy             | $6.423 \pm 0.150$ | $5.100 \pm 0.115$ | $4.306 \pm 0.098$ | $3.782 \pm 0.081$ | $3.440 \pm 0.072$ | $3.177 \pm 0.067$ | $1.440 \pm 0.022$ |         |
| Sliced Wasserstein | $6.556 \pm 0.148$ | $5.279 \pm 0.124$ | $4.529 \pm 0.105$ | $3.958 \pm 0.093$ | $3.578 \pm 0.083$ | $3.271 \pm 0.077$ | $1.494 \pm 0.025$ |         |
| Flow matching      | $6.311 \pm 0.141$ | $5.172 \pm 0.113$ | $4.462 \pm 0.100$ | $3.944 \pm 0.088$ | $3.606 \pm 0.076$ | $3.393 \pm 0.074$ | $2.539 \pm 0.052$ |         |
